
---
# Drug Target Interaction Prediction: Fusion Predictor
---

## Table of Contents

- [1. Setting up the Environment](#1-setting-up-the-environment)
  - [1.1 Importing the Required Packages](#11-importing-the-required-packages)
  - [1.2 Disabling the HF Verification](#12-disabling-the-hf-verification)
  - [1.3 Detecting the GPU](#13-detecting-the-gpu)
  - [1.4 Defining the Directory Structure](#14-defining-the-directory-structure)
- [2. Preparing the Data](#2-preparing-the-data)
  - [2.1 Loading the Topological Embeddings](#21-loading-the-topological-embeddings)
  - [2.2 Loading the Semantic Embeddings](#22-loading-the-semantic-embeddings)
  - [2.3 Loading the Dataset](#23-loading-the-dataset)
  - [2.4 Merging and Creating the Final DataFrames](#24-merging-and-creating-the-final-dataframes)
  - [2.5 Making the Heterogenous Graph](#25-making-the-heterogenous-graph)
  - [2.6 Exploring the Split](#26-exploring-the-split)
  - [2.7 Wrapping the Dataset into DataLoaders](#27-wrapping-the-dataset-into-dataloaders)
  - [2.8 Seeing the final Data Statistics](#28-seeing-the-final-data-statistics)
- [3. Putting it all Together](#3-putting-it-all-together)
  - [3.1 Defining the Feature Fusion and other Layers](#31-defining-the-feature-fusion-and-other-layers)
  - [3.2 Building the Model](#32-building-the-model)
  - [3.3 Training and Evaluating the Model](#33-training-and-evaluating-the-model)

# 1. Setting up the Environment
---

## 1.1 Importing the Required Packages

In [37]:
import os
import re
import requests
import gc
from collections import Counter
from pathlib import Path
from tqdm import tqdm
from itertools import islice

import random
random.seed(42)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display  

from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, recall_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.cuda.amp import autocast

from torch_geometric.nn import SAGEConv, to_hetero, BatchNorm
from torch_geometric.loader import DataLoader 
from torch.utils.data import Dataset
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T

from transformers import T5Tokenizer, T5EncoderModel, AutoTokenizer, AutoModel

from huggingface_hub import configure_http_backend

from rdkit import Chem
from rdkit.Chem import Draw

import esm

from gtda.homology import CubicalPersistence
from gtda.diagrams import BettiCurve, PersistenceLandscape

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

## 1.2 Disabling the HF Verification

In [2]:
def backend_factory() -> requests.Session:
    session = requests.Session()
    session.verify = False
    return session

configure_http_backend(backend_factory=backend_factory)

## 1.3 Detecting the GPU

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:0


## 1.4 Defining the Directory Structure

In [4]:
working_dir = Path("/home/user/Music/dont_touch/dti")
data_dir = working_dir / "data"

dataset_dirs = {"biosnap_random": data_dir / "biosnap/random"}

out_dir_mol = working_dir / "images" / "molecules"
out_dir_mol.mkdir(parents=True, exist_ok=True)

out_dir_prot = working_dir / "images" / "contacts"
os.makedirs(out_dir_prot, exist_ok=True)

out_dir_emb = working_dir/ "embeddings"
out_dir_emb.mkdir(parents=True, exist_ok=True)

out_dir_emb_s = out_dir_emb / "structural" 
out_dir_emb_s.mkdir(parents=True, exist_ok=True)

out_dir_emb_s_drug = out_dir_emb / "structural" / "drugs"
out_dir_emb_s_drug.mkdir(parents=True, exist_ok=True)

out_dir_emb_s_prot = out_dir_emb / "structural" / "proteins"
out_dir_emb_s_prot.mkdir(parents=True, exist_ok=True)

out_dir_emb_se = out_dir_emb / "semantic" 
out_dir_emb_se.mkdir(parents=True, exist_ok=True)

out_dir_emb_se_drug = out_dir_emb / "semantic" / "drugs"
out_dir_emb_se_drug.mkdir(parents=True, exist_ok=True)

out_dir_emb_se_prot = out_dir_emb / "semantic" / "proteins"
out_dir_emb_se_prot.mkdir(parents=True, exist_ok=True)

dataset = "biosnap_random"

# 2. Preparing the Data
---

## 2.1 Loading the Topological Embeddings

In [23]:
drug_indexes = np.load(out_dir_emb_s_drug / "molecule_names.npy")
molecule_structural_embedding = np.load(out_dir_emb_s_drug / "molecule_image_embeddings.npy")
df_drug_emb_s = pd.DataFrame({"drug_index": drug_indexes, "drug_structural_embedding": list(molecule_structural_embedding)})

protein_indexes = np.load(out_dir_emb_s_prot / "protein_index.npy")
protein_structural_embedding = np.load(out_dir_emb_s_prot / "protein_contact_embeddings.npy")
df_prot_emb_s = pd.DataFrame({"protein_index":  protein_indexes, "protein_structural_embedding": list(protein_structural_embedding)})

## 2.2 Loading the Semantic Embeddings

In [24]:
smile_names = np.load(out_dir_emb_se_drug / f"{dataset}_smiles.npy", allow_pickle=True)
can_smile_names = np.load(out_dir_emb_se_drug / f"{dataset}_canonical_smiles.npy", allow_pickle=True)
drug_embeddings_se = np.load(out_dir_emb_se_drug / f"{dataset}_drug_embeddings.npy", allow_pickle=True)

sequences_names = np.load(out_dir_emb_se_prot / f"{dataset}_target_sequences.npy", allow_pickle=True)
prot_embeddings_se = np.load(out_dir_emb_se_prot / f"{dataset}_sequence_embeddings.npy", allow_pickle=True)

df_drug_emb_se = pd.DataFrame({
    'smiles': smile_names,
    'can_smiles': can_smile_names,
    'drug_semantic_embedding': drug_embeddings_se.tolist()  
})

drugs_llm = df_drug_emb_se.dropna().reset_index(drop=True)

df_prot_emb_se = pd.DataFrame({
    'sequences': sequences_names,
    'protein_semantic_embedding': prot_embeddings_se.tolist()  
})

## 2.3 Loading the Dataset

In [45]:
def load_dataset(dataset):
    data_dir = working_dir / "data"

    dataset_dirs = {
        "biosnap_random": data_dir / "biosnap/random",
        "human_random": data_dir / "human/random",
        "human_cold": data_dir / "human/cold"}  

    dataset_path = dataset_dirs[dataset]    

    train_path = dataset_path / "train.csv"
    val_path = dataset_path / "val.csv"
    test_path = dataset_path / "test.csv"  

    train = pd.read_csv(train_path)
    valid = pd.read_csv(val_path)
    test = pd.read_csv(test_path)

    train['Set'] = "Train"
    valid['Set'] = "Valid"
    test['Set'] = "Test"

    dataset_data = pd.concat([train, valid, test], ignore_index=True)

    return dataset_data

## 2.4 Merging and Creating the Final DataFrames

In [46]:
drugs_idx = df_drug_emb_se.reset_index()
drugs_idx['index'] = drugs_idx['index'].astype(int)
df_drug_emb_s['drug_index'] = df_drug_emb_s['drug_index'].astype(int)
drugs = pd.merge(drugs_idx, df_drug_emb_s, left_on = 'index', right_on ='drug_index')
drugs.drop(columns=['index', 'drug_index', 'can_smiles'], inplace = True)

prot_idx = df_prot_emb_se.reset_index()
prot_idx['index'] = prot_idx['index'].astype(int)
df_prot_emb_s['protein_index'] = df_prot_emb_s['protein_index'].astype(int)
proteins = pd.merge(prot_idx,  df_prot_emb_s, left_on = 'index', right_on ='protein_index')
proteins.drop(columns=['index', 'protein_index'], inplace = True) 

dataset_data = load_dataset('biosnap_random')

drug_data = pd.merge(dataset_data, drugs, left_on='SMILES', right_on='smiles', how='inner')   
drug_data = drug_data.drop(columns=['smiles'])

df = pd.merge(drug_data, proteins, left_on='Protein', right_on='sequences', how='inner')
df = df.drop(columns=['sequences'])

## 2.5 Making the Heterogenous Graph

In [48]:
unique_drug_id = drugs['smiles'].unique()
unique_gene_id = proteins['sequences'].unique()

drug_id_map = {drug: idx for idx, drug in enumerate(unique_drug_id)}
gene_id_map = {gene: idx for idx, gene in enumerate(unique_gene_id)}

drug_semantic_embeddings = np.array([np.array(emb) for emb in drugs['drug_semantic_embedding']])
drug_structural_embeddings = np.array([np.array(emb) for emb in drugs['drug_structural_embedding']])
protein_semantic_embeddings = np.array([np.array(emb) for emb in proteins['protein_semantic_embedding']])
protein_structural_embeddings = np.array([np.array(emb) for emb in proteins['protein_structural_embedding']])

data = HeteroData()

data["drug"].node_id = torch.arange(len(unique_drug_id))
data["gene"].node_id = torch.arange(len(unique_gene_id))

data["drug"].xl = torch.tensor(drug_semantic_embeddings, dtype=torch.float)
data["drug"].xs = torch.tensor(drug_structural_embeddings, dtype=torch.float)
data["gene"].xl = torch.tensor(protein_semantic_embeddings, dtype=torch.float)
data["gene"].xs = torch.tensor(protein_structural_embeddings, dtype=torch.float)

edge_indices, edge_labels, edge_splits = [], [], []

for _, row in df.iterrows():
    drug_idx = drug_id_map[row["SMILES"]]
    gene_idx = gene_id_map[row["Protein"]]
    edge_indices.append([drug_idx, gene_idx])
    edge_labels.append(row["Y"])
    edge_splits.append(row["Set"])

edge_indices = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
edge_labels = torch.tensor(edge_labels, dtype=torch.float)

data["drug", "interacts_with", "gene"].edge_index = edge_indices
data["drug", "interacts_with", "gene"].edge_label = edge_labels

edge_splits = np.array(edge_splits)
train_mask = torch.tensor(edge_splits == "Train", dtype=torch.bool)
valid_mask = torch.tensor(edge_splits == "Valid", dtype=torch.bool)
test_mask = torch.tensor(edge_splits == "Test", dtype=torch.bool)

data["drug", "interacts_with", "gene"].train_mask = train_mask
data["drug", "interacts_with", "gene"].valid_mask = valid_mask
data["drug", "interacts_with", "gene"].test_mask = test_mask
data = T.ToUndirected()(data)

In [49]:
data

HeteroData(
  drug={
    node_id=[2400],
    xl=[2400, 768],
    xs=[2400, 1200],
  },
  gene={
    node_id=[2181],
    xl=[2181, 1024],
    xs=[2181, 1200],
  },
  (drug, interacts_with, gene)={
    edge_index=[2, 18348],
    edge_label=[18348],
    train_mask=[18348],
    valid_mask=[18348],
    test_mask=[18348],
  },
  (gene, rev_interacts_with, drug)={
    edge_index=[2, 18348],
    edge_label=[18348],
    train_mask=[18348],
    valid_mask=[18348],
    test_mask=[18348],
  }
)

## 2.6 Exploring the Split

In [50]:
print("\n" + "=" * 50)
print("             📝 Full Dataset")
print("=" * 50)
print(f"Drug nodes: {data['drug'].num_nodes}")
print(f"Gene nodes: {data['gene'].num_nodes}")
print(f"Edges:      {data['drug', 'interacts_with', 'gene'].num_edges}")

train_mask = data["drug", "interacts_with", "gene"].train_mask
print("\n" + "-" * 50)
print("           🚀 Training Dataset")
print("-" * 50)
print(f"Drug nodes: {data['drug'].num_nodes}")
print(f"Gene nodes: {data['gene'].num_nodes}")
print(f"Edges:      {train_mask.sum().item()}")

valid_mask = data["drug", "interacts_with", "gene"].valid_mask
print("\n" + "-" * 50)
print("         🔍 Validation Dataset")
print("-" * 50)
print(f"Drug nodes: {data['drug'].num_nodes}")
print(f"Gene nodes: {data['gene'].num_nodes}")
print(f"Edges:      {valid_mask.sum().item()}")

test_mask = data["drug", "interacts_with", "gene"].test_mask
print("\n" + "-" * 50)
print("             🎯 Test Dataset")
print("-" * 50)
print(f"Drug nodes: {data['drug'].num_nodes}")
print(f"Gene nodes: {data['gene'].num_nodes}")
print(f"Edges:      {test_mask.sum().item()}")


             📝 Full Dataset
Drug nodes: 2400
Gene nodes: 2181
Edges:      18348

--------------------------------------------------
           🚀 Training Dataset
--------------------------------------------------
Drug nodes: 2400
Gene nodes: 2181
Edges:      13294

--------------------------------------------------
         🔍 Validation Dataset
--------------------------------------------------
Drug nodes: 2400
Gene nodes: 2181
Edges:      1679

--------------------------------------------------
             🎯 Test Dataset
--------------------------------------------------
Drug nodes: 2400
Gene nodes: 2181
Edges:      3375


## 2.7 Wrapping the Dataset into DataLoaders

In [51]:
class EdgeDataset(Dataset):
    def __init__(self, data: HeteroData, mask: torch.Tensor):
        self.edge_index = data["drug", "interacts_with", "gene"].edge_index[:, mask]
        self.edge_label = data["drug", "interacts_with", "gene"].edge_label[mask]
        self.num_edges = self.edge_index.size(1)

    def __len__(self):
        return self.num_edges

    def __getitem__(self, idx):
        return self.edge_index[:, idx], self.edge_label[idx]


def custom_collate_fn(batch):
    edge_indices = [item[0] for item in batch]  
    edge_labels = [item[1] for item in batch]  
    
    edge_index = torch.cat(edge_indices, dim=1)  
    edge_label = torch.cat(edge_labels, dim=0)   

    return edge_index, edge_label


train_dataset = EdgeDataset(data, train_mask)
val_dataset = EdgeDataset(data, valid_mask)
test_dataset = EdgeDataset(data, test_mask)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, collate_fn=custom_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, collate_fn=custom_collate_fn)

## 2.8 Seeing the final Data Statistics

In [54]:
def count_edges(edge_labels):
    counter = Counter(edge_labels.tolist())
    return counter[1.0], counter[0.0]

train_labels = train_dataset.edge_label
valid_labels = val_dataset.edge_label
test_labels = test_dataset.edge_label

print("🔢 Total number of nodes:")
print(f"💊 Drugs: {data['drug'].xs.size(0)}")
print(f"🧬 Genes: {data['gene'].xs.size(0)}")
print()
print("$" * 60)
print()

train_positive, train_negative = count_edges(train_labels)
val_positive, val_negative = count_edges(valid_labels)
test_positive, test_negative = count_edges(test_labels)

print(" Edge Label Counts:")
print(f"🔶 Train - Total: {train_labels.size(0)}, Positive: {train_positive} , Negative: {train_negative} ")
print(f"🔶 Valid - Total: {valid_labels.size(0)}, Positive: {val_positive} , Negative: {val_negative} ")
print(f"🔶 Test  - Total: {test_labels.size(0)}, Positive: {test_positive} , Negative: {test_negative} ")

🔢 Total number of nodes:
💊 Drugs: 2400
🧬 Genes: 2181

$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

 Edge Label Counts:
🔶 Train - Total: 13294, Positive: 7255 , Negative: 6039 
🔶 Valid - Total: 1679, Positive: 961 , Negative: 718 
🔶 Test  - Total: 3375, Positive: 1904 , Negative: 1471 


In [55]:
drug_l_input_size = data['drug']['xl'].size(1)
gene_l_input_size = data['gene']['xl'].size(1)
drug_s_input_size = data['drug']['xs'].size(1)
gene_s_input_size = data['gene']['xs'].size(1)

print(f"🔷 Drug LLM Input Size: {drug_l_input_size}")
print(f"🧬 Gene LLM Input Size: {gene_l_input_size}")
print(f"💊 Drug Structure Input Size: {drug_s_input_size}")
print(f"🧪 Gene Structure Input Size: {gene_s_input_size}")


🔷 Drug LLM Input Size: 768
🧬 Gene LLM Input Size: 1024
💊 Drug Structure Input Size: 1200
🧪 Gene Structure Input Size: 1200


# 3. Putting it all Together
---

## 3.1 Defining the Feature Fusion and other Layers

In [ ]:
class FeatureFusion(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.fc = torch.nn.Linear(2 * hidden_channels, hidden_channels)

    def forward(self, llm_features, struct_features):
        concat_features = torch.cat([llm_features, struct_features], dim=-1)
        alpha = torch.sigmoid(self.fc(concat_features)) 
        return alpha * llm_features + (1 - alpha) * struct_features 


class Classifier(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.fc1 = torch.nn.Linear(2 * hidden_channels, hidden_channels)
        self.fc2 = torch.nn.Linear(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(0.5)

    def forward(self, drug_emb, gene_emb, edge_index):
        if edge_index.shape[0] != 2:
            edge_index = edge_index.t()

        src, dst = edge_index
        drug_emb_src = drug_emb[src]
        gene_emb_dst = gene_emb[dst]

        edge_emb = torch.cat([drug_emb_src, gene_emb_dst], dim=-1)
        edge_emb = F.relu(self.fc1(edge_emb))
        edge_emb = self.dropout(edge_emb)
        return self.fc2(edge_emb).squeeze(-1)


class GNN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv(hidden_channels, hidden_channels)
        self.bn1 = BatchNorm(hidden_channels)
        self.dropout = torch.nn.Dropout(0.5)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm(hidden_channels)
        

    def forward(self, x, edge_index):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        return x

## 3.2 Building the Model

In [ ]:
class Model(torch.nn.Module):
    def __init__(self, hidden_channels, drug_l_input_size, gene_l_input_size, drug_s_input_size, gene_s_input_size):
        super().__init__()


        self.drug_llm_lin = torch.nn.Sequential(
            torch.nn.Linear(drug_l_input_size, 512),
            torch.nn.BatchNorm1d(512),
            torch.nn.ReLU(),
            torch.nn.Linear(512, hidden_channels),
            torch.nn.BatchNorm1d(hidden_channels),
            torch.nn.ReLU()
        )
        self.drug_struct_lin = torch.nn.Sequential(
            torch.nn.Linear(drug_s_input_size, 1024),
            torch.nn.BatchNorm1d(1024),
            torch.nn.ReLU(),
            torch.nn.Linear(1024, hidden_channels),
            torch.nn.BatchNorm1d(hidden_channels),
            torch.nn.ReLU()
        )


        self.gene_llm_lin = torch.nn.Sequential(
            torch.nn.Linear(gene_l_input_size, 1024),
            torch.nn.BatchNorm1d(1024),
            torch.nn.ReLU(),
            torch.nn.Linear(1024, hidden_channels),
            torch.nn.BatchNorm1d(hidden_channels),
            torch.nn.ReLU()
        )
        self.gene_struct_lin = torch.nn.Sequential(
            torch.nn.Linear(gene_s_input_size, 1024),
            torch.nn.BatchNorm1d(1024),
            torch.nn.ReLU(),
            torch.nn.Linear(1024, hidden_channels),
            torch.nn.BatchNorm1d(hidden_channels),
            torch.nn.ReLU()
        )

        self.feature_fusion = FeatureFusion(hidden_channels)
        self.gnn = GNN(hidden_channels)
        self.gnn = to_hetero(self.gnn, metadata=data.metadata())
        self.classifier = Classifier(hidden_channels)

    def forward(self, data, edge_index, edge_label):
        drug_xl = data["drug"].xl.to(device).float()
        drug_xs = data["drug"].xs.to(device).float()
        gene_xl = data["gene"].xl.to(device).float()
        gene_xs = data["gene"].xs.to(device).float()

        drug_xs = (drug_xs - drug_xs.mean(dim=0)) / (drug_xs.std(dim=0, unbiased=False) + 1e-8)
        gene_xs = (gene_xs - gene_xs.mean(dim=0)) / (gene_xs.std(dim=0, unbiased=False) + 1e-8)

        drug_llm = self.drug_llm_lin(drug_xl)
        drug_struct = self.drug_struct_lin(drug_xs)
        drug_x = self.feature_fusion(drug_llm, drug_struct)

        gene_llm = self.gene_llm_lin(gene_xl)
        gene_struct = self.gene_struct_lin(gene_xs)
        gene_x = self.feature_fusion(gene_llm, gene_struct)

        x_dict = {"drug": drug_x, "gene": gene_x}
        x_dict = self.gnn(x_dict, data.edge_index_dict)

        drug_emb = x_dict["drug"]
        gene_emb = x_dict["gene"]
        pred = self.classifier(drug_emb, gene_emb, edge_index)

        return pred, edge_label

## 3.3 Training and Evaluating the Model

In [59]:
model_save_dir = os.path.join(working_dir, "models", dataset)
os.makedirs(model_save_dir, exist_ok=True)


def train_and_validate(model, data, train_loader, val_loader, device, patience=5, max_epochs=100, dataset_name="default", run_id=1):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data.edge_index_dict = {key: edge_index.to(device) for key, edge_index in data.edge_index_dict.items()}
    model = model.to(device)
    
    all_labels = torch.cat([labels for _, labels in train_loader], dim=0)
    negative_to_positive_ratio = len(all_labels[all_labels == 0]) / len(all_labels[all_labels == 1])
    pos_weight = torch.tensor(negative_to_positive_ratio).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=4, eta_min=1e-6)
    
    best_auroc = 0.0
    best_val_loss = float("inf")
    epochs_no_improve = 0
    
    for epoch in range(1, max_epochs + 1):
        model.train()
        total_train_loss, total_train_examples = 0, 0
        
        for edge_index, edge_label in train_loader:
            optimizer.zero_grad()
            edge_index, edge_label = edge_index.to(device), edge_label.to(device)
            
            pred, ground_truth = model(data, edge_index, edge_label)
            loss = F.binary_cross_entropy_with_logits(pred, ground_truth, pos_weight)
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item() * pred.numel()
            total_train_examples += pred.numel()
        
        avg_train_loss = total_train_loss / total_train_examples
        
        model.eval()
        total_val_loss, total_val_examples = 0, 0
        preds, ground_truths = [], []
        
        with torch.no_grad():
            for edge_index, edge_label in val_loader:
                edge_index, edge_label = edge_index.to(device), edge_label.to(device)
                pred, ground_truth = model(data, edge_index, edge_label)
                loss = F.binary_cross_entropy_with_logits(pred, ground_truth)
                
                total_val_loss += loss.item() * pred.numel()
                total_val_examples += pred.numel()
                
                preds.append(pred.cpu())
                ground_truths.append(ground_truth.cpu())
        
        avg_val_loss = total_val_loss / total_val_examples
        preds = torch.cat(preds, dim=0).sigmoid().numpy()
        ground_truths = torch.cat(ground_truths, dim=0).numpy()
        
        auroc = roc_auc_score(ground_truths, preds)
        
        if auroc > best_auroc:
            best_auroc = auroc
            best_epoch = epoch
            model_save_path = os.path.join(model_save_dir, f"{dataset_name}_best_model_run_{run_number+1}.pt")
            torch.save(model.state_dict(), model_save_path)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve >= patience:
            break
    
    return best_auroc


def evaluate_test_set(model, data, data_loader, device):
    model.eval()
    preds, ground_truths = [], []
    
    with torch.no_grad():
        for edge_index, edge_label in data_loader:
            edge_index, edge_label = edge_index.to(device), edge_label.to(device)
            pred, _ = model(data, edge_index, edge_label)
            preds.append(pred.cpu())
            ground_truths.append(edge_label.cpu())
    
    preds = torch.cat(preds, dim=0).sigmoid().numpy()
    ground_truths = torch.cat(ground_truths, dim=0).numpy()
    
    binary_preds = (preds > 0.6).astype(int)
    
    auroc = roc_auc_score(ground_truths, preds)
    aupr = average_precision_score(ground_truths, preds)
    accuracy = accuracy_score(ground_truths, binary_preds)
    sensitivity = recall_score(ground_truths, binary_preds)
    tn, fp, fn, tp = confusion_matrix(ground_truths, binary_preds).ravel()
    specificity = tn / (tn + fp)
    
    return auroc, aupr, accuracy, sensitivity, specificity


hidden_channels = 512
model = Model(hidden_channels, drug_l_input_size, gene_l_input_size, drug_s_input_size, gene_s_input_size)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

auroc_scores, aupr_scores = [], []
accuracy_scores, sensitivity_scores, specificity_scores = [], [], []

for run_number in range(5):
    gc.collect()
    torch.cuda.empty_cache()
    seed = random.randint(0, 2**32 - 1)
    torch.manual_seed(seed)
    
    model = Model(hidden_channels, drug_l_input_size, gene_l_input_size, drug_s_input_size, gene_s_input_size)
    
    train_and_validate(
        model=model,
        data=data,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        dataset_name='biosnap_random',
        run_id=run_number
    )
    
    model_path = working_dir / f"models/biosnap_random/biosnap_random_best_model_run_{run_number+1}.pt"
    model.load_state_dict(torch.load(model_path, map_location=device))
    
    auroc, aupr, accuracy, sensitivity, specificity = evaluate_test_set(model, data, test_loader, device)
    
    auroc_scores.append(auroc)
    aupr_scores.append(aupr)
    accuracy_scores.append(accuracy)
    sensitivity_scores.append(sensitivity)
    specificity_scores.append(specificity)
    
    print(f"🔥 Run {run_number+1}: AUROC = {auroc:.4f}, AUPRC = {aupr:.4f}, Accuracy = {accuracy:.4f}, Sensitivity = {sensitivity:.4f}, Specificity = {specificity:.4f}")


print("\n🏆 Final Results:")
print(f"🔹 AUROC: {np.mean(auroc_scores):.4f}")
print(f"🔹 AUPRC: {np.mean(aupr_scores):.4f}")
print(f"🔹 Accuracy: {np.mean(accuracy_scores):.4f}")
print(f"🔹 Sensitivity: {np.mean(sensitivity_scores):.4f}")
print(f"🔹 Specificity: {np.mean(specificity_scores):.4f}")

🔥 Run 1: AUROC = 0.9359, AUPRC = 0.9521, Accuracy = 0.8536, Sensitivity = 0.8745, Specificity = 0.8266
🔥 Run 2: AUROC = 0.9342, AUPRC = 0.9500, Accuracy = 0.8519, Sensitivity = 0.8424, Specificity = 0.8640
🔥 Run 3: AUROC = 0.9342, AUPRC = 0.9520, Accuracy = 0.8542, Sensitivity = 0.8356, Specificity = 0.8783
🔥 Run 4: AUROC = 0.9342, AUPRC = 0.9508, Accuracy = 0.8513, Sensitivity = 0.8514, Specificity = 0.8511
🔥 Run 5: AUROC = 0.9323, AUPRC = 0.9487, Accuracy = 0.8539, Sensitivity = 0.8676, Specificity = 0.8362

🏆 Final Results:
🔹 AUROC: 0.9341
🔹 AUPRC: 0.9507
🔹 Accuracy: 0.8530
🔹 Sensitivity: 0.8543
🔹 Specificity: 0.8513
